# 01 Data Audit — PhysioNet/CinC Challenge 2019
Audits the downloaded `.psv` tree: file counts per environment, schema, missingness, sepsis prevalence, and record integrity. Writes `docs/DATA_NOTES.md` and a download manifest.

In [ ]:
import os
from pathlib import Path
from vist_nrf.data.download import audit_tree, verify_counts, save_manifest
from vist_nrf.data.physionet2019 import list_psv_files, infer_hospital_system, audit_psv

root = os.environ.get('VIST_NRF_DATA', '/content/drive/MyDrive/VIST-NRF/data')
raw_root = Path(root) / 'raw' / 'physionet2019'
print('Raw root:', raw_root, '| exists:', raw_root.exists())

if raw_root.exists():
    files = list_psv_files(raw_root)
    print('Total .psv files:', len(files))
    from collections import Counter
    envs = Counter(infer_hospital_system(p) for p in files)
    print('Per-environment:', dict(envs))
    manifest = audit_tree(raw_root, source_url='https://registry.opendata.aws/challenge-2019/')
    problems = verify_counts(manifest)
    if problems:
        print('PROBLEMS:'); [print(' -', p) for p in problems]
    else:
        print('Counts match spec (A=20336, B=20000)')
    save_manifest(raw_root, Path(root) / 'download_manifest.json')
    print('Manifest saved.')
else:
    print('No data downloaded yet. See docs/DATA_NOTES.md for the aws s3 sync command.')